In [1]:
# 0. Environment fix (fresh machine only), then Runtime > Restart session
!pip uninstall -y torchao
!pip install -q --no-deps bitsandbytes

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
# 1. Setup: GPU check, Drive, data (validation rows for a loading check, test rows for the final run), tokenizer
import torch
from importlib.metadata import version, PackageNotFoundError

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU attached. Use Runtime > Change runtime type > T4 GPU, then run this cell again."
print("GPU:", torch.cuda.get_device_name(0))

for lib in ["torchao", "bitsandbytes", "peft", "transformers"]:
    try:
        print(lib, version(lib))
    except PackageNotFoundError:
        print(lib, "NOT installed")

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from transformers import AutoTokenizer

DATA_DIR = "/content/drive/MyDrive/promt shield"
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAXLEN = 256

df = pd.read_csv(f"{DATA_DIR}/dataset_v2.csv", dtype=str, keep_default_na=False)
assert set(df["split"]) == {"train", "val", "test"}
df["y"] = (df["final_label"] == "attack").astype(int)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
print("val rows:", len(val_df), "| test rows:", len(test_df))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print("pad token id:", tok.pad_token_id, "| padding side:", tok.padding_side)

torch: 2.14.1+cu130 | CUDA available: True
GPU: Tesla T4
torchao NOT installed
bitsandbytes 0.50.2
peft 0.21.2
transformers 5.22.1
Mounted at /content/drive
val rows: 3298 | test rows: 3300


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

pad token id: 151643 | padding side: right


In [3]:
# 2. Imports and settings
import os
import gc
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

device = torch.device("cuda")
RESULTS_DIR = f"{DATA_DIR}/results"
CKPT_DIR = f"{DATA_DIR}/checkpoints"
print("GPU memory in use at the start (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

GPU memory in use at the start (GB): 0.0


In [4]:
# 3. Functions: batching, scoring, loading, and the loading check
def make_batch(id_lists, pad_id):
    seqs = [torch.tensor(x) for x in id_lists]
    lengths = torch.tensor([len(s) for s in seqs])
    input_ids = pad_sequence(seqs, batch_first=True, padding_value=pad_id)
    attention_mask = (torch.arange(input_ids.shape[1])[None, :] < lengths[:, None]).long()
    return input_ids.to(device), attention_mask.to(device)


def forward_probs(model, id_lists, pad_id, batch_size):
    probs = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(id_lists), batch_size):
            ids, mask = make_batch(id_lists[i:i + batch_size], pad_id)
            logits = model(input_ids=ids, attention_mask=mask).logits
            probs.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
    return np.array(probs)


def tokenize_all(texts):
    return [tok(t[:4000], truncation=True, max_length=MAXLEN)["input_ids"] for t in texts]


def load_lora(path):
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, dtype=torch.float16)
    base.config.pad_token_id = tok.pad_token_id
    return PeftModel.from_pretrained(base.to(device), path), tok


def load_qlora(path):
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, quantization_config=bnb_cfg, dtype=torch.float16, device_map={"": 0},
    )
    base.config.pad_token_id = tok.pad_token_id
    return PeftModel.from_pretrained(base, path), tok


def check_loading(model, saved_val_csv, n=200):
    saved = pd.read_csv(saved_val_csv).set_index("id")["prob_attack"]
    sub = val_df.iloc[:n]
    now = forward_probs(model, tokenize_all(sub["text"]), tok.pad_token_id, 32)
    diff = np.abs(now - saved.loc[sub["id"]].values)
    print(f"   loading check on {n} validation prompts: mean abs difference {diff.mean():.4f} | max {diff.max():.4f}")
    assert diff.mean() < 0.02 and diff.max() < 0.3, (
        "The checkpoint does not reproduce the saved validation probabilities. Do not run the test.")

In [5]:
# 4. Score the test split once and save the probabilities (no metrics are computed here)
def run_final(tag, build, saved_val_csv):
    model, _ = build()
    model.eval()
    check_loading(model, saved_val_csv)
    probs = forward_probs(model, tokenize_all(test_df["text"]), tok.pad_token_id, 32)
    out = test_df[["id", "source_dataset", "y"]].copy()
    out["prob_attack"] = probs
    out.to_csv(f"{RESULTS_DIR}/{tag}_test_probs.csv", index=False)
    print(tag, "| test prompts scored:", len(out), "| all probabilities finite:", bool(np.isfinite(probs).all()),
          "| score range:", round(float(probs.min()), 3), "to", round(float(probs.max()), 3))
    del model
    gc.collect()
    torch.cuda.empty_cache()


run_final("lora", lambda: load_lora(f"{CKPT_DIR}/lora_epoch2"), f"{RESULTS_DIR}/lora_val_probs_epoch2.csv")
run_final("qlora", lambda: load_qlora(f"{CKPT_DIR}/qlora_epoch1"), f"{RESULTS_DIR}/qlora_val_probs_epoch1.csv")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   loading check on 200 validation prompts: mean abs difference 0.0000 | max 0.0000
lora | test prompts scored: 3300 | all probabilities finite: True | score range: 0.0 to 1.0


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   loading check on 200 validation prompts: mean abs difference 0.0000 | max 0.0000
qlora | test prompts scored: 3300 | all probabilities finite: True | score range: 0.0 to 1.0
